# 4 — Tune and verify a primitive resonance

> **Lesson focus**
>
> **Learn:** distinguish a component parameter from an active search
> variable. **Run:** tune the primitive capacitor toward 6.2 GHz.
> **Inspect:** the best ParameterSet and its explicit resonance
> readback. **Status:** `CONVERGING` scaffold.

## Declare the search, not a new circuit

The capacitor retained from lesson 1 exposes its bindable value through
`parameter("capacitance")`. The resulting `ParameterRef` identifies that
physical value. Only `OptimizationVariable` activates it and supplies
finite bounds. This separation is the [parameter-versus-variable
contract](../../docs/concepts/units-parameters-and-optimization.qmd#parameter-vs-optimization-variable);
it does not require a `CompositePlan`.

`CostObjective` compares a typed quantity with a target after relative
normalization. `CMAESSpec` supplies execution controls, and
`OptimizationSpec` binds the variables, objectives, and optimizer into
one inspectable request.

In [ ]:
from fixtures.primitive_resonator import build_primitive_resonator
from scnsim import (
    CMAESSpec,
    CircuitRun,
    CostObjective,
    DiagonalRootSpec,
    OptimizationSpec,
    OptimizationVariable,
    ReductionPipeline,
    units as u,
)

fixture = build_primitive_resonator()
run = CircuitRun(plan=fixture.plan, workspace="workspaces/primitive-course")
view = run.original.reduce(ReductionPipeline().retain(fixture.resonator_node))
resonator_capacitance = fixture.resonator_cap.parameter("capacitance")
root_spec = DiagonalRootSpec(
    coordinate=fixture.resonator_node,
    root_hint=6.0 * u.GHz,
)
optimization_spec = OptimizationSpec(
    variables=(
        OptimizationVariable(
            parameter=resonator_capacitance,
            bounds=(80.0 * u.fF, 140.0 * u.fF),
        ),
    ),
    objectives=(
        CostObjective(
            id="resonance_frequency",
            quantity=root_spec.frequency,
            target=6.2 * u.GHz,
            weight=1.0 * u.dimensionless,
        ),
    ),
    optimizer=CMAESSpec(seed=17, max_evaluations=200),
)
optimization_spec.show()

`root_spec.frequency` is a typed quantity selector used to define the
objective; accessing it does not execute. By contrast, `root.frequency`
on an evaluated Result is a materialized physical Quantity.

## Execute once and keep the winner explicit

Candidate binding, quantity evaluation, objective aggregation, and
CMA-ES stay inside one Julia process. The returned best parameters are
immutable data, not mutable state on the Run.

In [ ]:
optimization = run.optimize(view, optimization_spec)
optimization.best.parameters
optimization.show()

winner_root = run.evaluate(
    view,
    root_spec,
    parameters=optimization.best.parameters,
)
winner_root.frequency
winner_root.show()

The winner remains an explicit immutable `ParameterSet`. Passing it
creates a new exact request; it does not mutate the Plan baseline or the
View.

[Previous](03_evaluate_quantity.qmd) · [Course
map](../../docs/index.qmd) · [Next: report and
resolve](05_report_resolve.qmd) · [Concept: units and
optimization](../../docs/concepts/units-parameters-and-optimization.qmd#parameter-vs-optimization-variable)